In [61]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
#from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [62]:
data = pd.read_csv("../data/stud.csv")

In [63]:
X = data.drop(columns=["math_score"], axis=1)
y = data["math_score"]

In [64]:
numerical_cols = X.select_dtypes(exclude="object").columns
categorical_cols = X.select_dtypes(include="object").columns

In [65]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
transformers = []
if len(categorical_cols) > 0: transformers.append(("categorical transform", OneHotEncoder(), categorical_cols))
if len(numerical_cols) > 0: transformers.append(("numerical transform", StandardScaler(), numerical_cols))
ct = ColumnTransformer(transformers= transformers)

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [67]:
def adj_r2(true, predicted, n_features):
    r2 = r2_score(true, predicted)
    n = len(true)
    
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

In [68]:
def evaluate_model(true, predicted, n_features = None):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    adj_r2_sc = adj_r2(true, predicted, n_features)
    return mae, rmse, r2_square, adj_r2_sc

In [69]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    #"CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

In [70]:
for name, model in models.items():
   
    model.fit(X_train, y_train)
    n_features = X_train.shape[1]
    train_pred = model.predict(X_train)
    train_mae, train_rmse, train_r2, train_adj_r2 = evaluate_model(y_train, train_pred, n_features)
    
    test_pred = model.predict(X_test)
    test_mae, test_rmse, test_r2, test_adj_r2 = evaluate_model(y_test, test_pred, n_features)

    model_list.append(name)
    
    print(name)
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))
    print("- Adj. R2 Score: {:.4f}".format(train_adj_r2))


    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))
    print("- Adj. R2 Score: {:.4f}".format(test_adj_r2))

    r2_list.append(test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3231
- Mean Absolute Error: 4.2667
- R2 Score: 0.8743
- Adj. R2 Score: 0.8713
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3940
- Mean Absolute Error: 4.2148
- R2 Score: 0.8804
- Adj. R2 Score: 0.8678


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5925
- Mean Absolute Error: 5.2053
- R2 Score: 0.8072
- Adj. R2 Score: 0.8025
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5173
- Mean Absolute Error: 5.1557
- R2 Score: 0.8254
- Adj. R2 Score: 0.8070


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3233
- Mean Absolute Error: 4.2650
- R2 Score: 0.8743
- Adj. R2 Score: 0.8712
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3904
- Mean Absolute Error: 4.2111
- R2 Score: 0.8806
- Adj. R2 Score: 0.8680


K-Neighbors Regress

In [71]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
2,Ridge,0.880592
0,Linear Regression,0.880433
5,Random Forest Regressor,0.852136
7,AdaBoost Regressor,0.849308
6,XGBRegressor,0.827797
1,Lasso,0.825446
3,K-Neighbors Regressor,0.785944
4,Decision Tree,0.728033
